# 03 – Preprocessing & Feature Engineering

Baut Lags, Rolling-Fenster und Kalender-Features. **Train + Test werden zusammengefügt**, damit Lags am Test-Anfang die Train-Historie nutzen.

| Modus | Wo | Daten | RAM |
|-------|-----|--------|-----|
| `sample` | Lokal | `train_sample.csv` + `test.csv` | ~500 MB |
| `full` | **Colab (CPU)** | voller `train.csv` — **streaming** pro Region | ~2–4 GB RAM |

**Outputs** (für Notebook 04):
- `outputs/processed/train_labeled.parquet` – nur Zeilen mit `score`
- `outputs/processed/test_features.parquet` – alle Test-Zeilen

Siehe [docs/06_EDA_ANALYSIS.md](../docs/06_EDA_ANALYSIS.md) für die Feature-Logik.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import pyarrow  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyarrow"])

# Projekt-Setup + Import folgt in Zelle 1 (nach Drive-Mount in Colab)
print("Basis-Imports OK — als Nächstes Zelle 1 ausführen")

## 1. Umgebung & Modus

In [ ]:
try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False


def find_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    if IS_COLAB:
        candidates += [
            Path("/content/DataMining_Final-Project"),
            Path("/content/drive/MyDrive/DataMining/DataMining_Final-Project"),
        ]
    for p in candidates:
        if (p / "scripts" / "features.py").exists():
            return p.resolve()
    return Path.cwd().resolve()


if IS_COLAB:
    drive.mount("/content/drive")

PROJECT_ROOT = find_project_root()

# Colab: fester Drive-Pfad (deine Struktur)
COLAB_ROOT = Path("/content/drive/MyDrive/DataMining/DataMining_Final-Project")
if IS_COLAB and (COLAB_ROOT / "scripts" / "features.py").exists():
    PROJECT_ROOT = COLAB_ROOT.resolve()

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

def resolve_data_dir(root: Path) -> Path:
    for candidate in [
        root / "data",
        Path("/content/drive/MyDrive/DataMining/DataMining_Final-Project/data"),
        Path("/content/drive/MyDrive/DataMining/data"),
    ]:
        if (candidate / "train.csv").exists():
            return candidate
    return root / "data"

DATA_DIR = resolve_data_dir(PROJECT_ROOT)
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

# sample = lokal testen | full = Colab mit vollem Train
MODE = "full" if IS_COLAB else "sample"
# MODE = "sample"  # zum manuellen Überschreiben

if MODE == "sample":
    TRAIN_PATH = DATA_DIR / "train_sample.csv"
    if not TRAIN_PATH.exists():
        raise FileNotFoundError("train_sample.csv fehlt — python scripts/create_sample.py")
else:
    TRAIN_PATH = DATA_DIR / "train.csv"

print(f"Umgebung: {'Colab' if IS_COLAB else 'Lokal'} | Modus: {MODE}")
print(f"  TRAIN: {TRAIN_PATH}")
print(f"  TEST:  {TEST_PATH}")
for p, name in [(TRAIN_PATH, "Train"), (TEST_PATH, "Test")]:
    print(f"  {'✓' if p.exists() else '✗'} {name}: {p}")

PROCESSED_DIR = PROJECT_ROOT / "outputs" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

features_path = PROJECT_ROOT / "scripts" / "features.py"
print("cwd:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)
print("features.py:", features_path, "exists:", features_path.exists())

if not features_path.exists():
    raise FileNotFoundError(f"{features_path} nicht gefunden — scripts/ auf Drive prüfen.")

import importlib.util

_spec = importlib.util.spec_from_file_location("dm_features", features_path)
_features = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_features)
build_features = _features.build_features
feature_columns = _features.feature_columns
add_persistence_baseline = _features.add_persistence_baseline
print("features geladen OK")

## 2. Features berechnen & speichern

- **`sample`:** alles im RAM (klein)
- **`full`:** **Streaming pro Region** (~2–4 GB RAM, dauert ~20–40 Min.)

In [ ]:
path_train_std = PROCESSED_DIR / "train_labeled.parquet"
path_test_std = PROCESSED_DIR / "test_features.parquet"
feat_cols = feature_columns()

if MODE == "full":
    # Speicherschonend: nie 12M Zeilen auf einmal im RAM
    import importlib.util

    _p = PROJECT_ROOT / "scripts" / "preprocess_streaming.py"
    _spec = importlib.util.spec_from_file_location("preprocess_streaming", _p)
    _ps = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_ps)

    print("Streaming-Modus (pro Region, ~2–4 GB RAM) — bitte warten …")
    stats = _ps.preprocess_by_region(
        TRAIN_PATH,
        TEST_PATH,
        path_train_std,
        path_test_std,
        chunk_size=500_000,
    )
    print("Fertig:", stats)
    train_labeled = pd.read_parquet(path_train_std).head(5)  # nur Vorschau
    display(train_labeled)
else:
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)
    train["_split"] = "train"
    test["_split"] = "test"
    test["score"] = np.nan
    panel = pd.concat([train, test], ignore_index=True)
    del train, test

    panel = build_features(panel)
    panel = add_persistence_baseline(panel, lag_days=7)

    train_feat = panel[panel["_split"] == "train"]
    test_feat = panel[panel["_split"] == "test"]
    train_labeled = train_feat[train_feat["score"].notna()]

    meta_cols = ["region_id", "date", "year", "month", "day", "ordinal", "score", "score_persist7"]
    save_train_cols = [c for c in list(dict.fromkeys(meta_cols + feat_cols)) if c in train_labeled.columns]
    save_test_cols = [c for c in list(dict.fromkeys(
        ["region_id", "date", "year", "month", "day", "ordinal"] + feat_cols
    )) if c in test_feat.columns]

    train_labeled[save_train_cols].to_parquet(path_train_std, index=False)
    test_feat[save_test_cols].to_parquet(path_test_std, index=False)
    print(f"Gespeichert: {len(train_labeled):,} train | {len(test_feat):,} test")

## 5. Sanity-Check

In [ ]:
import pyarrow.parquet as pq
n_train = pq.read_metadata(path_train_std).num_rows
n_test = pq.read_metadata(path_test_std).num_rows
print(f"Parquet: train_labeled {n_train:,} | test {n_test:,}")

preview = pd.read_parquet(path_train_std).head(2000)
missing_feat = preview[feat_cols].isna().mean().sort_values(ascending=False).head(10)
print("Top fehlende Features (wegen Lags am Serienanfang):")
display(missing_feat)

print("\nScore-Verteilung (labeled train):")
print(preview["score"].describe())